# Geospatial ML — Train on Colab & Save to Drive

This notebook trains:
1. **ResNet-50 classifier** on EuroSAT (land-cover classification)
2. **Convolutional Autoencoder** for anomaly detection

Checkpoints are saved to your Google Drive under `MyDrive/geospatial_checkpoints/`.

> Make sure **Runtime → Change runtime type → T4 GPU** is selected before running.

## Step 1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_OUTPUT = '/content/drive/MyDrive/geospatial_checkpoints'
os.makedirs(DRIVE_OUTPUT, exist_ok=True)
print(f'Checkpoints will be saved to: {DRIVE_OUTPUT}')

## Step 2 — Clone the repo

In [ ]:
%cd /content
!git clone https://github.com/iamvisheshsrivastava/geospatial
%cd geospatial
!git log --oneline -3

## Step 3 — Install dependencies

In [ ]:
!pip install -q \
    torch torchvision \
    rasterio \
    wandb \
    boto3 \
    scikit-learn \
    numpy pandas matplotlib pillow \
    pydantic pydantic-settings \
    tqdm

import torch
print(f'PyTorch {torch.__version__}')
print(f'GPU available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## Step 4 — Download EuroSAT dataset (~90 MB)

Uses `wget` with `--no-check-certificate` to avoid SSL issues with the DFKI server.

In [ ]:
import os

os.makedirs('data/eurosat', exist_ok=True)

# Download via wget (bypasses SSL certificate verification issues on Colab)
!wget -q --no-check-certificate \
    'https://madm.dfki.de/files/sentinel/EuroSAT.zip' \
    -O data/eurosat/EuroSAT.zip

print('Download complete. Extracting...')

import zipfile, shutil
from pathlib import Path

with zipfile.ZipFile('data/eurosat/EuroSAT.zip', 'r') as zf:
    zf.extractall('data/eurosat')

# Flatten the nested folder structure
for candidate in ['data/eurosat/EuroSAT/2750', 'data/eurosat/EuroSAT']:
    p = Path(candidate)
    if p.exists():
        for cls_dir in p.iterdir():
            dest = Path('data/eurosat') / cls_dir.name
            if dest.exists():
                shutil.rmtree(dest)
            shutil.move(str(cls_dir), str(dest))
        shutil.rmtree(p.parent if '2750' in candidate else p, ignore_errors=True)
        break

Path('data/eurosat/EuroSAT.zip').unlink(missing_ok=True)

classes = ['AnnualCrop','Forest','HerbaceousVegetation','Highway','Industrial',
           'Pasture','PermanentCrop','Residential','River','SeaLake']
for cls in classes:
    d = Path(f'data/eurosat/{cls}')
    count = len(list(d.glob('*'))) if d.exists() else 0
    status = 'OK' if d.exists() else 'MISSING'
    print(f'  {status:7s}  {cls} ({count} images)')

## Step 5 — Train the ResNet-50 classifier

~10 minutes on T4 GPU. Trains for 10 epochs with full fine-tuning.

In [ ]:
!python -m src.train \
    --data-root data/eurosat \
    --epochs 10 \
    --batch-size 64 \
    --learning-rate 3e-4 \
    --num-workers 2 \
    --checkpoint-dir checkpoints \
    --wandb-mode disabled

## Step 6 — Train the Anomaly Detector (Autoencoder)

~10 minutes on T4 GPU. Trains on the Forest class only.

In [ ]:
!python -m src.anomaly \
    --data-root data/eurosat \
    --normal-classes Forest \
    --epochs 30 \
    --wandb-mode disabled

## Step 7 — Verify checkpoints were created

In [ ]:
import os
for f in ['checkpoints/best_model.pt', 'checkpoints/autoencoder_best.pt']:
    if os.path.exists(f):
        size_mb = os.path.getsize(f) / 1_048_576
        print(f'  OK  {f}  ({size_mb:.1f} MB)')
    else:
        print(f'  MISSING  {f} — check training output above for errors')

## Step 8 — Copy checkpoints to Google Drive

In [ ]:
import shutil, os

files_to_save = [
    'checkpoints/best_model.pt',
    'checkpoints/autoencoder_best.pt',
    'checkpoints/training_history.json',
]

for src in files_to_save:
    if os.path.exists(src):
        dst = os.path.join(DRIVE_OUTPUT, os.path.basename(src))
        shutil.copy2(src, dst)
        size_mb = os.path.getsize(dst) / 1_048_576
        print(f'  Saved  {dst}  ({size_mb:.1f} MB)')
    else:
        print(f'  SKIPPED (not found): {src}')

print('\nDone! Download these files from your Google Drive.')

## Step 9 — Quick sanity check (optional)

Loads the classifier and runs a prediction on a random EuroSAT image to confirm the model works.

In [ ]:
import torch
from pathlib import Path
from src.models.resnet import build_resnet50_classifier
from src.data.preprocessing import preprocess_image

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ckpt = torch.load('checkpoints/best_model.pt', map_location=device)
class_names = ckpt['class_names']
model = build_resnet50_classifier(num_classes=len(class_names), pretrained=False)
model.load_state_dict(ckpt['model_state_dict'])
model.to(device).eval()

# Grab a random image from the dataset
sample = next(Path('data/eurosat').glob('*/*.jpg'))
tensor = preprocess_image(sample, 224).unsqueeze(0).to(device)

with torch.no_grad():
    probs = torch.softmax(model(tensor), dim=1).squeeze()

conf, idx = probs.max(0)
print(f'Image : {sample}')
print(f'Predicted : {class_names[idx]} ({conf:.1%} confidence)')
print(f'Val macro F1: {ckpt["metrics"]["macro_f1"]:.4f}')